In [ ]:
import requests
from bs4 import BeautifulSoup
import re

base_url = 'https://www.bassamfattouh.com/en/views/ajax?_wrapper_format=drupal_ajax&view_name=shop&view_display_id=page_1&view_args=1&view_path=%2Fshop&view_base_path=shop&view_dom_id=c1c26311759959edb4d2b9997014c7e00c55ba8836e27c224578ea34502c042a&pager_element=0&page={page_num}&_drupal_ajax=1&ajax_page_state%5Btheme%5D=bassam&ajax_page_state%5Btheme_token%5D=&ajax_page_state%5Blibraries%5D=eJyFU22y2yAMvBA2R2IEKJgGkEeIJK-nr0he8tp8TP_I0moxsFrgF1xcIYjIFn7yVTYm75GNh96hWq8kXRj2O3CgJgucsVPFO5YKeShLl6-SW7qjveRwNB5FkB1eduoY3SEXLbuFIeT68DXLJ0rChgzFhAJMz3sEKsRKxRLtNV-u-XIgrjB_t_QzSNiTzQnJABcDIc_B-IJ7oY7EjxRHGMjJVA4Li7Tv7IdhfSWOPgI_SDS3mBxs13H5J_pjlAQOk28tjBrLdqDRuGo6fLcs4xobwnbVnvy2HLQW__lnHKeO7LuJdJRKmgE0g2aXiu11dSNAVaGpBUCxpN-Ms1TCQZJFOnECQfcFF1j6ZC7tRU3hvqiGJ0OqwrndmBIakbtsdJfpB1t334kvuG0XScd3GwdzdHrTquBQXvB9z0jIPfOt9ZO8wt9EJORfBMm_3fp1B6YMT2vj8z9V3_1FUP5qmHEe8qJqjYhlVH4z_AtKHa4zoPe403wf8Gqlqk4A26Pi9Xp1OeanNGP01rv78rFtQtZI0okEtfO5z-TxJKOuiPtIr6ItNrf4YXUDZdtF5f3R8J8KIj'

all_products = []
current_page = 0
total_products_available = float('inf') # Initialize with infinity to ensure the loop runs

while True:
    url = base_url.format(page_num=current_page)
    print(f"Fetching page {current_page}...")

    response = requests.get(url)
    response.raise_for_status() # Raise an exception for HTTP errors
    json_response = response.json()

    html_data = None
    for item in json_response:
        if item.get('command') == 'viewsLoadMoreAppend' and item.get('data'):
            html_data = item.get('data')
            break
        # For the first page, the command might be 'insert' with method 'replaceWith'
        elif current_page == 0 and item.get('command') == 'insert' and item.get('method') == 'replaceWith' and item.get('data'):
            html_data = item.get('data')
            break

    if not html_data:
        print("No more HTML data found in response. Exiting loop.")
        break

    soup = BeautifulSoup(html_data, 'html.parser')

    # Extract total product count from header on the first page or if it changes
    if current_page == 0:
        header_span = soup.select_one('.view-header-products')
        if header_span:
            header_text = header_span.get_text() # e.g., "Showing 1 - 16 of 70 products"
            match = re.search(r'of (\d+) products', header_text)
            if match:
                total_products_available = int(match.group(1))
                print(f"Total products available: {total_products_available}")
            else:
                print("Could not parse total products from header.")
        else:
            print("Header span for total products not found.")

    product_elements = soup.select('.shop-product')
    if not product_elements:
        print(f"No products found on page {current_page}. Exiting loop.")
        break

    current_page_products = []
    for product_element in product_elements:
        product_name_element = product_element.select_one('img')
        product_name = product_name_element['alt'] if product_name_element and 'alt' in product_name_element.attrs else 'N/A'

        product_link_element = product_element.select_one('a')
        product_link = product_link_element['href'] if product_link_element and 'href' in product_link_element.attrs else 'N/A'
        if product_link and product_link.startswith('/'):
            product_link = 'https://www.bassamfattouh.com' + product_link

        image_url_element = product_element.select_one('img')
        image_url = image_url_element['src'] if image_url_element and 'src' in image_url_element.attrs else 'N/A'
        if image_url and image_url.startswith('/'):
            image_url = 'https://www.bassamfattouh.com' + image_url

        current_page_products.append({
            'name': product_name,
            'link': product_link,
            'image_url': image_url
        })

    all_products.extend(current_page_products)
    print(f"Fetched {len(current_page_products)} products from page {current_page}. Total unique products gathered so far: {len(all_products)}")

    # Check for termination condition based on the number of products
    if len(current_page_products) == 0 or len(all_products) >= total_products_available:
        print("Reached end of products or fetched all available products. Exiting loop.")
        break

    current_page += 1

print(f"Finished fetching. Total products collected: {len(all_products)}")

Fetching page 0...
Total products available: 70
Fetched 16 products from page 0. Total unique products gathered so far: 16
Fetching page 1...
Fetched 16 products from page 1. Total unique products gathered so far: 32
Fetching page 2...
Fetched 16 products from page 2. Total unique products gathered so far: 48
Fetching page 3...
Fetched 16 products from page 3. Total unique products gathered so far: 64
Fetching page 4...
Fetched 6 products from page 4. Total unique products gathered so far: 70
Reached end of products or fetched all available products. Exiting loop.
Finished fetching. Total products collected: 70


In [ ]:
import json
# Assuming all_products is populated by the previous cell
if 'all_products' in locals() or 'all_products' in globals():
    print(json.dumps(all_products, indent=2))
else:
    print("No products data available. Run the previous cell to fetch data.")

[
  {
    "name": "2 in 1 Pencil Glo* Frosted shadow",
    "link": "https://www.bassamfattouh.com/en/shop/category/eyes-114/2-1-pencils-all-over-165?v=536",
    "image_url": "https://www.bassamfattouh.com/sites/default/files/styles/web_product/public/products/48542scr_285595f7884e053.jpg.webp?itok=bRf_ljqs"
  },
  {
    "name": "4 Items Brush Kit",
    "link": "https://www.bassamfattouh.com/en/shop/category/brushes-117/4-items-brush-kit-173?v=550",
    "image_url": "https://www.bassamfattouh.com/sites/default/files/styles/web_product/public/products/41565scr_80b807ab6d38f6e.jpg.webp?itok=FQ3o6u-d"
  },
  {
    "name": "Aiguille Liner Pen Matte Eyeliner Noir ",
    "link": "https://www.bassamfattouh.com/en/shop/category/eyes-114/aiguille-liner-pen-matte-eyeliner-noir-159?v=526",
    "image_url": "https://www.bassamfattouh.com/sites/default/files/styles/web_product/public/products/41456scr_4fb29685b4009c1.jpg.webp?itok=O4AIHYYN"
  },
  {
    "name": "<table border=\"0\" cellpadding=\"0\"

In [ ]:
print("Product parsing logic has been moved to the fetching cell (above). The full list of products is available in 'all_products' and printed by the cell directly below the fetching cell.")

Product parsing logic has been moved to the fetching cell (above). The full list of products is available in 'all_products' and printed by the cell directly below the fetching cell.
